# read data from github

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/<你的用户名>/<repo名>/main/Source/Data/processed/sentiment/gpt-5_And_Closing_Price.csv"

df = pd.read_csv(url)

# ARDL Model

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ardl import ARDL
from statsmodels.tsa.stattools import adfuller

# -----------------------------
# 1️⃣ 加载数据
# -----------------------------
DATA_PATH = "Source/Data/processed/sentiment/gpt-5_And_Closing_Price.csv"
df = pd.read_csv(DATA_PATH)

# 转为 datetime 并按日期排序
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")
df.index = pd.date_range(start=df.index[0], periods=len(df), freq='B')

# -----------------------------
# 2️⃣ 构造因变量 return（log-return 也可）
# -----------------------------
df["return"] = df["Close"].pct_change()
df = df.dropna()

# -----------------------------
# 3️⃣ 单位根检验，确保变量可用
# -----------------------------
adf_y = adfuller(df["return"])
print("Return ADF p-value:", adf_y[1])

adf_x = adfuller(df["Sentiment"])
print("Sentiment ADF p-value:", adf_x[1])

# -----------------------------
# 4️⃣ 准备建模变量
# -----------------------------
y = df["return"]
x = df[["Sentiment"]]

# -----------------------------
# 5️⃣ 自动选择 ARDL 滞后阶数（p,q）
# 这里示例：尝试 p=1~5, q=1~5
# -----------------------------
best_aic = np.inf
best_order = None
best_model = None

for p in range(1, 6):
    for q in range(1, 6):
        try:
            model = ARDL(y, lags=p, exog=x, order=q)
            result = model.fit()
            if result.aic < best_aic:
                best_aic = result.aic
                best_order = (p, q)
                best_model = result
        except Exception as e:
            continue

print(f"Best ARDL order selected by AIC: p={best_order[0]}, q={best_order[1]}")
print(best_model.summary())

# -----------------------------
# 6️⃣ 短期预测
# -----------------------------
# 预测未来 n 天 return
n_forecast = 1
forecast = best_model.forecast(steps=n_forecast, exog=x.iloc[-n_forecast:])
print("Forecasted returns for next", n_forecast, "days:")
print(forecast)

# 将 return 转为 Close 价格预测
last_close = df["Close"].iloc[-1]
predicted_close = [last_close * (1 + forecast.iloc[0])]
for r in forecast.iloc[1:]:
    predicted_close.append(predicted_close[-1] * (1 + r))

predicted_close = pd.Series(predicted_close, name="Predicted_Close")
print("Predicted Close Prices:")
print(predicted_close)


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ardl import ARDL
from statsmodels.tsa.stattools import adfuller

DATA_PATH = "Source/Data/processed/sentiment/gpt-5_And_Closing_Price.csv"
df = pd.read_csv(DATA_PATH)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# -----------------------------
# 1️⃣ 周频平均
# -----------------------------
df_weekly = df.resample('W-FRI').agg({
    'Close': 'last',      # 本周最后收盘价
    'Sentiment': 'mean'   # 本周平均 sentiment
})

# -----------------------------
# 2️⃣ 构造滞后 sentiment (0~3 lag)
# -----------------------------
# 处理缺失 sentiment
df_weekly['Sentiment'] = df_weekly['Sentiment'].fillna(method='ffill')  # 前值填充

# 构造滞后 sentiment
x = pd.DataFrame()
x['Sentiment_L0'] = df_weekly['Sentiment']
x['Sentiment_L1'] = df_weekly['Sentiment'].shift(1)
x['Sentiment_L2'] = df_weekly['Sentiment'].shift(2)
x['Sentiment_L3'] = df_weekly['Sentiment'].shift(3)

# 对齐 y 与 x，统一删除 NaN
data = pd.concat([df_weekly['Close'], x], axis=1).dropna()
y = data['Close']
x = data[['Sentiment_L0', 'Sentiment_L1', 'Sentiment_L2', 'Sentiment_L3']]

# -----------------------------
# 3️⃣ 单位根检验
# -----------------------------
print("ADF test for y (Close):", adfuller(y)[1])
for col in x.columns:
    print(f"ADF test for {col}:", adfuller(x[col])[1])

from statsmodels.tsa.ardl import ardl_select_order
sel = ardl_select_order(y, maxlag=3, maxorder=(1, 3), exog=x)
best_model = sel.model.fit()
print("Best Model:")
print(best_model.summary())

best_result = sel.model.fit()
# 用 best model 预测下一期
best_forecast = best_result.forecast(steps=1)
print("Best model predicted next week Close:", best_forecast.iloc[0])

# -----------------------------
# 4️⃣ 建 ARDL
# -----------------------------
model = ARDL(y, lags=1, exog=x, order=0)  # lags=1 表示 y 的滞后1期
result = model.fit()
print("lags1 Model:")
print(result.summary())

# -----------------------------
# 5️⃣ 预测下一周收盘价
# -----------------------------
# 准备 exog
exog_forecast = x.iloc[-1:].copy()  # 最新的 lag0~3
forecast = result.forecast(steps=1, exog=exog_forecast)
print("Predicted next week Close:", forecast.iloc[0])


In [ ]:
import pandas as pd
import numpy as np

from statsmodels.tsa.ardl import ardl_select_order
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# 0. 读数据
# -----------------------------
DATA_PATH = "Source/Data/processed/sentiment/gpt-5_And_Closing_Price.csv"

df = pd.read_csv(DATA_PATH)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# -----------------------------
# 1. 周频数据
# -----------------------------
df_weekly = df.resample('W-FRI').agg({
    'Close': 'last',
    'Sentiment': 'mean'
})

df_weekly['Sentiment'] = df_weekly['Sentiment'].ffill()

# 只保留需要的变量
data = df_weekly[['Close', 'Sentiment']].dropna()

y = data['Close']
X = data[['Sentiment']]

# -----------------------------
# 2. 单位根检验
# -----------------------------
print("ADF p-value (Close):", adfuller(y)[1])
print("ADF p-value (Sentiment):", adfuller(X['Sentiment'])[1])

# -----------------------------
# 3. 划分样本（预测评估用）
# -----------------------------
split = int(len(data) * 0.8)

y_train = y.iloc[:split]
y_test  = y.iloc[split:]

X_train = X.iloc[:split]
X_test  = X.iloc[split:]

# -----------------------------
# 4. ARDL order selection
# -----------------------------
sel = ardl_select_order(
    endog=y_train,
    exog=X_train,
    maxlag=3,      # y 的最大滞后
    maxorder=3,    # x 的最大滞后
    ic='aic',
    trend='c'
)

print(sel)

# -----------------------------
# 5. 估计 best ARDL
# -----------------------------
result = sel.model.fit()

print("\n================ ARDL result ================\n")
print(result.summary())

# =========================================================
# 1️⃣ 统计显著性：t-stat, p-value
# =========================================================
print("\n--- t-statistics ---")
print(result.tvalues)

print("\n--- p-values ---")
print(result.pvalues)

# =========================================================
# 2️⃣ 拟合优度
# =========================================================
print("\nR-squared:", result.rsquared)
print("Adj R-squared:", result.rsquared_adj)

# =========================================================
# 3️⃣ 信息准则
# =========================================================
print("\nAIC :", result.aic)
print("BIC :", result.bic)
print("HQIC:", result.hqic)

# =========================================================
# 4️⃣ Bounds test（协整）
# =========================================================
print("\n=========== Bounds F-test for cointegration ===========\n")

bounds_res = result.bounds_test(case=3)
print(bounds_res)

# =========================================================
# 5️⃣ ECM（误差修正模型）
# =========================================================
print("\n=========== ECM representation ===========\n")

ecm_res = result.ecm()

print(ecm_res.summary())

# ECM(-1) 系数
print("\nECM(-1) coefficient:")
print(ecm_res.params)

# =========================================================
# 6️⃣ 样本外预测（rolling dynamic forecast）
# =========================================================

# 用完整样本重新估计模型（用于预测）
final_sel = ardl_select_order(
    endog=y,
    exog=X,
    maxlag=3,
    maxorder=3,
    ic='aic',
    trend='c'
)

final_res = final_sel.model.fit()

# 预测测试集长度
forecast = final_res.forecast(
    steps=len(y_test),
    exog=X_test
)

# 对齐 index
forecast.index = y_test.index

# =========================================================
# 7️⃣ 预测误差
# =========================================================
rmse = np.sqrt(mean_squared_error(y_test, forecast))
mae  = mean_absolute_error(y_test, forecast)
mape = np.mean(np.abs((y_test - forecast) / y_test)) * 100

print("\n=========== Forecast performance ===========")
print("RMSE :", rmse)
print("MAE  :", mae)
print("MAPE :", mape)

# 如果你想看实际 vs 预测
out = pd.DataFrame({
    "Actual": y_test,
    "Forecast": forecast
})

print("\nActual vs Forecast (head):")
print(out.head())

# result = sel.model.fit()判断sentiment不是很重要 （Model:ARDL(2,)）
# 所以proceed不到

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ardl import ARDL, ardl_select_order
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# 0️⃣ 读数据
# -----------------------------
DATA_PATH = "Source/Data/processed/sentiment/gpt-5_And_Closing_Price.csv"
df = pd.read_csv(DATA_PATH)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# -----------------------------
# 1️⃣ 周频数据 & 收益率
# -----------------------------
df_weekly = df.resample('W-FRI').agg({
    'Close': 'last',
    'Sentiment': 'mean'
})
df_weekly['Sentiment'] = df_weekly['Sentiment'].ffill()  # if use weekly, go find why (use daily)
df_weekly['ret'] = np.log(df_weekly['Close']).diff() # if use return, go find whether any document
df_weekly = df_weekly.dropna()

# -----------------------------
# 2️⃣ 构造滞后 sentiment
# -----------------------------
for lag in range(5):
    df_weekly[f'Sentiment_L{lag}'] = df_weekly['Sentiment'].shift(lag)
df_weekly['Sentiment_SMA3'] = df_weekly['Sentiment'].rolling(3).mean()
df_weekly = df_weekly.dropna()

y = df_weekly['ret']
X = df_weekly[['Sentiment_L0','Sentiment_L1','Sentiment_L2','Sentiment_L3','Sentiment_L4','Sentiment_SMA3']]

# -----------------------------
# 3️⃣ 单位根检验
# -----------------------------
print("ADF p-value (ret):", adfuller(y)[1])
for col in X.columns:
    print(f"ADF p-value ({col}):", adfuller(X[col])[1])

# -----------------------------
# 4️⃣ ARDL 阶数选择
# -----------------------------
sel = ardl_select_order(endog=y, exog=X, maxlag=4, maxorder=4, ic='aic', trend='c')

# ⚠️ 在新版本 statsmodels 中 sel.model 已经是最佳模型
best_model = sel.model.fit()

print("\n======= ARDL Model Summary =======\n")
print(best_model.summary())

# -----------------------------
# 5️⃣ t-stat, p-value
# -----------------------------
print("\n--- t-statistics ---")
print(best_model.tvalues)
print("\n--- p-values ---")
print(best_model.pvalues)

# -----------------------------
# 6️⃣ 信息准则
# -----------------------------
print("\nAIC :", best_model.aic)
print("BIC :", best_model.bic)
print("HQIC:", best_model.hqic)

# -----------------------------
# 7️⃣ Bounds Test (长期协整)
# -----------------------------
try:
    bounds_res = best_model.bounds_test(case=3)
    print("\nBounds Test Result:")
    print(bounds_res)
except Exception as e:
    print("\nBounds Test not available:", e)

# -----------------------------
# 8️⃣ ECM 表示
# -----------------------------
try:
    ecm_res = best_model.ecm()
    print("\n======= ECM Summary =======\n")
    print(ecm_res.summary())
    print("\nECM(-1) coefficient:")
    print(ecm_res.params)
except Exception as e:
    print("\nECM not available:", e)

# -----------------------------
# 9️⃣ 样本外预测（最后 10% 数据）
# -----------------------------
split = int(len(y) * 0.9)
y_train, y_test = y.iloc[:split], y.iloc[split:]
X_train, X_test = X.iloc[:split], X.iloc[split:]

final_sel = ardl_select_order(endog=y_train, exog=X_train, maxlag=4, maxorder=4, ic='aic', trend='c')
final_model = final_sel.model.fit()

forecast = final_model.forecast(steps=len(y_test), exog=X_test)
forecast.index = y_test.index

# -----------------------------
# 10️⃣ 预测误差
# -----------------------------
rmse = np.sqrt(mean_squared_error(y_test, forecast))
mae  = mean_absolute_error(y_test, forecast)
mape = np.mean(np.abs((y_test - forecast) / y_test)) * 100

print("\n=========== Forecast performance ===========")
print("RMSE :", rmse)
print("MAE  :", mae)
print("MAPE :", mape)

out = pd.DataFrame({"Actual": y_test, "Forecast": forecast})
print("\nActual vs Forecast (head):")
print(out.head())

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ardl import ARDL, ardl_select_order
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -----------------------------
# 0️⃣ 读数据
# -----------------------------
DATA_PATH = "Source/Data/processed/sentiment/gpt-5_And_Closing_Price.csv"
df = pd.read_csv(DATA_PATH)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# -----------------------------
# 1️⃣ 周频数据 & 收益率
# -----------------------------
df_weekly = df.resample('W-FRI').agg({
    'Close': 'last',
    'Sentiment': 'mean'
})
df_weekly['Sentiment'] = df_weekly['Sentiment'].ffill()
df_weekly['ret'] = np.log(df_weekly['Close']).diff()
df_weekly = df_weekly.dropna()

# -----------------------------
# 2️⃣ 构造涨跌方向
# -----------------------------
df_weekly['Direction'] = (df_weekly['ret'] > 0).astype(int)

# -----------------------------
# 3️⃣ 构造滞后变量 (y 和 Sentiment)
# -----------------------------
for lag in range(1,4):  # 滞后 1~3 周
    df_weekly[f'ret_L{lag}'] = df_weekly['ret'].shift(lag)
    df_weekly[f'Sentiment_L{lag}'] = df_weekly['Sentiment'].shift(lag)

df_weekly = df_weekly.dropna()

y = df_weekly['Direction']
X = df_weekly[['ret_L1','ret_L2','ret_L3','Sentiment_L1','Sentiment_L2','Sentiment_L3']]

# -----------------------------
# 4️⃣ 划分训练/测试集
# -----------------------------
split = int(len(df_weekly) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# -----------------------------
# 5️⃣ ARDL 阶数选择
# -----------------------------
sel = ardl_select_order(endog=y_train, exog=X_train, maxlag=3, maxorder=3, ic='aic', trend='c')

# sel.model 已经是最佳 ARDL
best_model = sel.model.fit()
print("\n======= ARDL Model Summary =======\n")
print(best_model.summary())

# -----------------------------
# 6️⃣ 样本外预测（方向预测）
# -----------------------------
forecast_continuous = best_model.forecast(steps=len(y_test), exog=X_test)
# 将连续预测转换为 0/1
forecast_direction = (forecast_continuous > 0).astype(int)
forecast_direction.index = y_test.index

# -----------------------------
# 7️⃣ 预测性能
# -----------------------------
print("\n======= Forecast Performance =======")
print("Accuracy:", accuracy_score(y_test, forecast_direction))
print(classification_report(y_test, forecast_direction))
print("Confusion Matrix:")
print(confusion_matrix(y_test, forecast_direction))

# 查看实际 vs 预测
out = pd.DataFrame({
    "Actual": y_test,
    "Forecast": forecast_direction
})
print("\nActual vs Forecast (head):")
print(out)

# -------------------------------------------------------------